# Post-Processing & Visualization

## Overview

Post-processing is a critical step in Finite Element Analysis that transforms numerical results into meaningful physical insights. This notebook covers comprehensive visualization and analysis techniques for electromagnetic FEA results, including field visualization, energy calculations, force computations, and comparative analysis across different solvers.

## Learning Objectives

After completing this notebook, you will be able to:
- Create effective visualizations of electromagnetic fields
- Compute derived quantities (energy, forces, flux)
- Perform comparative analysis across different solutions
- Generate publication-quality plots and animations
- Extract engineering-relevant metrics from FEA results

## Topics Covered

1. **Field Visualization Techniques** (contours, vectors, streamlines)
2. **Derived Quantity Computation** (energy, flux, forces)
3. **Comparative Analysis** between different solvers/methods
4. **Advanced Visualization** (3D plots, animations)
5. **Engineering Metrics** extraction and analysis

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import spsolve
from scipy.interpolate import griddata
import matplotlib.tri as tri
from matplotlib.patches import Circle, Rectangle, Polygon, FancyArrowPatch
from matplotlib.collections import PatchCollection, LineCollection
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import warnings
warnings.filterwarnings('ignore')

# Set up matplotlib for better plots
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=4, suppress=True)

# Advanced plotting setup
plt.style.use('default')
colors_palette = plt.cm.Set3(np.linspace(0, 1, 12))

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Available colormaps: {len(plt.colormaps())}")

## 1. Field Visualization Techniques

Effective visualization of electromagnetic fields is crucial for understanding physical behavior. We'll explore various techniques including contour plots, vector fields, streamlines, and combination visualizations.

In [ ]:
# ---------- 1. Advanced Field Visualization ----------
def create_test_field_data(n_points=1000, domain_size=2.0):
    """
    Create synthetic electromagnetic field data for visualization demonstration.
    
    Returns:
    --------
    points, elements, A_z, Bx, By : arrays
        Mesh coordinates, connectivity, and field data
    """
    # Create mesh
    np.random.seed(42)
    
    # Generate points in circular domain
    points = []
    
    # Center region (dense)
    n_center = 400
    for i in range(n_center):
        r = domain_size * 0.3 * np.sqrt(np.random.rand())
        theta = np.random.rand() * 2 * np.pi
        points.append([r * np.cos(theta), r * np.sin(theta)])
    
    # Middle region
    n_middle = 400
    for i in range(n_middle):
        r = domain_size * 0.3 + domain_size * 0.4 * np.sqrt(np.random.rand())
        theta = np.random.rand() * 2 * np.pi
        points.append([r * np.cos(theta), r * np.sin(theta)])
    
    # Outer region
    n_outer = 200
    for i in range(n_outer):
        r = domain_size * 0.7 + domain_size * 0.3 * np.sqrt(np.random.rand())
        theta = np.random.rand() * 2 * np.pi
        points.append([r * np.cos(theta), r * np.sin(theta)])
    
    # Add boundary points
    n_boundary = 60
    for i in range(n_boundary):
        theta = 2 * np.pi * i / n_boundary
        points.append([domain_size * np.cos(theta), domain_size * np.sin(theta)])
    
    points = np.array(points)
    
    # Create triangulation
    tri_obj = Delaunay(points)
    elements = tri_obj.simplices
    
    # Create synthetic field data
    # Magnetic vector potential A_z (dipole-like field)
    source_pos = np.array([0.3, 0.2])
    sink_pos = np.array([-0.3, -0.2])
    
    A_z = np.zeros(len(points))
    for i, point in enumerate(points):
        r_source = np.linalg.norm(point - source_pos)
        r_sink = np.linalg.norm(point - sink_pos)
        
        # Dipole-like potential
        if r_source > 0.01:
            A_z[i] += 0.5 * np.log(r_source + 0.1)
        if r_sink > 0.01:
            A_z[i] -= 0.5 * np.log(r_sink + 0.1)
        
        # Add some asymmetry
        A_z[i] += 0.1 * point[0] * point[1]
    
    # Compute magnetic field B = curl(A)
    Bx = np.zeros_like(A_z)
    By = np.zeros_like(A_z)
    
    for elem in elements:
        coords = points[elem]
        x = coords[:, 0]
        y = coords[:, 1]
        
        # Area
        A_e = 0.5 * abs(np.linalg.det(np.array([
            [1, x[0], y[0]],
            [1, x[1], y[1]],
            [1, x[2], y[2]]
        ])))
        
        if A_e > 1e-10:
            # Gradients
            b = np.array([y[1] - y[2], y[2] - y[0], y[0] - y[1]])
            c = np.array([x[2] - x[1], x[0] - x[2], x[1] - x[0]])
            
            dA_dx = np.dot(A_z[elem], b) / (2 * A_e)
            dA_dy = np.dot(A_z[elem], c) / (2 * A_e)
            
            Bx[elem] += dA_dy / 3
            By[elem] -= dA_dx / 3
    
    return points, elements, A_z, Bx, By

# Generate test data
points_viz, elements_viz, A_z_viz, Bx_viz, By_viz = create_test_field_data()
B_magnitude_viz = np.sqrt(Bx_viz**2 + By_viz**2)

print(f"Test field data created:")
print(f"  Points: {len(points_viz)}")
print(f"  Elements: {len(elements_viz)}")
print(f"  A_z range: [{A_z_viz.min():.3f}, {A_z_viz.max():.3f}]")
print(f"  |B| range: [{B_magnitude_viz.min():.3f}, {B_magnitude_viz.max():.3f}]")

In [ ]:
# ---------- 1.1 Contour Plot Variations ----------
def create_advanced_contour_plots(points, elements, A_z, B_magnitude):
    """
    Create various contour plot styles for field visualization.
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Advanced Contour Visualization Techniques', fontsize=16)

    # 1. Standard filled contour
    ax = axes[0, 0]
    levels_A = np.linspace(A_z.min(), A_z.max(), 20)
    contour1 = ax.tricontourf(points[:, 0], points[:, 1], elements, A_z, 
                             levels=levels_A, cmap='viridis')
    fig.colorbar(contour1, ax=ax, label='A_z [Wb/m]')
    ax.set_title('Standard Filled Contour')
    ax.set_aspect('equal')
    
    # 2. Contour lines
    ax = axes[0, 1]
    contour2 = ax.tricontour(points[:, 0], points[:, 1], elements, A_z, 
                            levels=15, colors='black', linewidths=0.5)
    ax.clabel(contour2, inline=True, fontsize=8)
    contour2_fill = ax.tricontourf(points[:, 0], points[:, 1], elements, A_z, 
                                  levels=levels_A, cmap='viridis', alpha=0.3)
    ax.set_title('Contour Lines with Labels')
    ax.set_aspect('equal')
    
    # 3. Magnetic field magnitude
    ax = axes[0, 2]
    levels_B = np.linspace(0, B_magnitude.max(), 20)
    contour3 = ax.tricontourf(points[:, 0], points[:, 1], elements, B_magnitude, 
                             levels=levels_B, cmap='plasma')
    fig.colorbar(contour3, ax=ax, label='|B| [T]')
    ax.set_title('Magnetic Field Magnitude')
    ax.set_aspect('equal')
    
    # 4. Logarithmic scale
    ax = axes[1, 0]
    B_log = np.log10(B_magnitude + 1e-10)
    levels_log = np.linspace(B_log.min(), B_log.max(), 20)
    contour4 = ax.tricontourf(points[:, 0], points[:, 1], elements, B_log, 
                             levels=levels_log, cmap='hot')
    fig.colorbar(contour4, ax=ax, label='log₁₀(|B|)')
    ax.set_title('Logarithmic Scale')
    ax.set_aspect('equal')
    
    # 5. Gradient magnitude
    ax = axes[1, 1]
    # Compute gradient magnitude
    grad_magnitude = np.zeros(len(points))
    for elem in elements:
        coords = points[elem]
        x = coords[:, 0]
        y = coords[:, 1]
        
        A_e = 0.5 * abs(np.linalg.det(np.array([
            [1, x[0], y[0]],
            [1, x[1], y[1]],
            [1, x[2], y[2]]
        ])))
        
        if A_e > 1e-10:
            b = np.array([y[1] - y[2], y[2] - y[0], y[0] - y[1]])
            c = np.array([x[2] - x[1], x[0] - x[2], x[1] - x[0]])
            
            grad_x = np.dot(A_z[elem], b) / (2 * A_e)
            grad_y = np.dot(A_z[elem], c) / (2 * A_e)
            grad_mag = np.sqrt(grad_x**2 + grad_y**2)
            
            grad_magnitude[elem] += grad_mag / 3
    
    levels_grad = np.linspace(0, grad_magnitude.max(), 20)
    contour5 = ax.tricontourf(points[:, 0], points[:, 1], elements, grad_magnitude, 
                             levels=levels_grad, cmap='coolwarm')
    fig.colorbar(contour5, ax=ax, label='|∇A_z|')
    ax.set_title('Gradient Magnitude')
    ax.set_aspect('equal')
    
    # 6. Multi-field overlay
    ax = axes[1, 2]
    # Background: A_z contours
    contour6_bg = ax.tricontourf(points[:, 0], points[:, 1], elements, A_z, 
                                levels=levels_A, cmap='viridis', alpha=0.5)
    
    # Overlay: High B regions
    B_threshold = B_magnitude.max() * 0.7
    high_B_mask = B_magnitude > B_threshold
    
    if np.any(high_B_mask):
        ax.scatter(points[high_B_mask, 0], points[high_B_mask, 1], 
                  c=B_magnitude[high_B_mask], cmap='Reds', s=20, alpha=0.7,
                  label='High |B| regions')
    
    ax.set_title('Multi-Field Overlay')
    ax.set_aspect('equal')
    ax.legend()
    
    # Add domain boundary to all plots
    for ax in axes.flat:
        boundary_circle = Circle((0, 0), 2.0, fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(boundary_circle)
        ax.set_xlim(-2.2, 2.2)
        ax.set_ylim(-2.2, 2.2)
        ax.set_xlabel('x [m]')
        ax.set_ylabel('y [m]')
    
    plt.tight_layout()
    return fig

# Create contour visualizations
fig_contours = create_advanced_contour_plots(points_viz, elements_viz, A_z_viz, B_magnitude_viz)
plt.show()

print("Advanced contour visualizations created!")

In [ ]:
# ---------- 1.2 Vector Field Visualization ----------
def create_vector_field_plots(points, elements, Bx, By, B_magnitude):
    """
    Create various vector field visualization styles.
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Vector Field Visualization Techniques', fontsize=16)

    # Create interpolation grid for vector plots
    xi = np.linspace(-2.0, 2.0, 40)
    yi = np.linspace(-2.0, 2.0, 40)
    Xi, Yi = np.meshgrid(xi, yi)
    
    # Interpolate fields to grid
    Bx_interp = griddata(points, Bx, (Xi, Yi), method='linear', fill_value=0)
    By_interp = griddata(points, By, (Xi, Yi), method='linear', fill_value=0)
    B_mag_interp = griddata(points, B_magnitude, (Xi, Yi), method='linear', fill_value=0)
    
    # Mask points outside domain
    mask = np.sqrt(Xi**2 + Yi**2) > 2.0
    Bx_interp[mask] = np.nan
    By_interp[mask] = np.nan
    B_mag_interp[mask] = np.nan

    # 1. Standard quiver plot
    ax = axes[0, 0]
    # Background contours
    contour_bg1 = ax.tricontourf(points[:, 0], points[:, 1], elements, B_magnitude, 
                                levels=20, cmap='viridis', alpha=0.3)
    
    # Vector field
    skip = 2
    quiver1 = ax.quiver(Xi[::skip, ::skip], Yi[::skip, ::skip], 
                       Bx_interp[::skip, ::skip], By_interp[::skip, ::skip],
                       B_mag_interp[::skip, ::skip], cmap='plasma', 
                       scale=np.max(B_mag_interp)*25, width=0.003)
    fig.colorbar(quiver1, ax=ax, label='|B| [T]')
    ax.set_title('Standard Quiver Plot')
    ax.set_aspect('equal')
    
    # 2. Streamlines
    ax = axes[0, 1]
    # Background magnitude
    contour_bg2 = ax.contourf(Xi, Yi, B_mag_interp, levels=20, cmap='viridis', alpha=0.5)
    
    # Streamlines
    stream = ax.streamplot(Xi, Yi, Bx_interp, By_interp, 
                          color='white', density=1.5, linewidth=1, 
                          arrowsize=1.2, cmap='autumn')
    ax.set_title('Magnetic Field Streamlines')
    ax.set_aspect('equal')
    
    # 3. Combined quiver and streamlines
    ax = axes[0, 2]
    contour_bg3 = ax.tricontourf(points[:, 0], points[:, 1], elements, B_magnitude, 
                                levels=20, cmap='coolwarm', alpha=0.4)
    
    # Streamlines
    ax.streamplot(Xi, Yi, Bx_interp, By_interp, 
                 color='black', density=1.0, linewidth=0.8, 
                 arrowsize=1.0, alpha=0.6)
    
    # Selective vectors (high magnitude regions)
    high_B_mask_grid = B_mag_interp > (B_magnitude.max() * 0.5)
    if np.any(high_B_mask_grid):
        ax.quiver(Xi[high_B_mask_grid], Yi[high_B_mask_grid], 
                 Bx_interp[high_B_mask_grid], By_interp[high_B_mask_grid],
                 B_mag_interp[high_B_mask_grid], cmap='Reds', 
                 scale=np.max(B_mag_interp)*20, width=0.004, alpha=0.8)
    
    ax.set_title('Streamlines + Selective Vectors')
    ax.set_aspect('equal')
    
    # 4. Vector magnitude with direction indicators
    ax = axes[1, 0]
    contour_mag = ax.contourf(Xi, Yi, B_mag_interp, levels=20, cmap='plasma')
    fig.colorbar(contour_mag, ax=ax, label='|B| [T]')
    
    # Direction indicators (unit vectors)
    B_unit_x = Bx_interp / (B_mag_interp + 1e-10)
    B_unit_y = By_interp / (B_mag_interp + 1e-10)
    
    # Plot unit vectors with fixed length
    skip_dir = 3
    ax.quiver(Xi[::skip_dir, ::skip_dir], Yi[::skip_dir, ::skip_dir], 
             B_unit_x[::skip_dir, ::skip_dir], B_unit_y[::skip_dir, ::skip_dir],
             scale=20, width=0.002, color='white', alpha=0.7)
    
    ax.set_title('Field Magnitude + Direction')
    ax.set_aspect('equal')
    
    # 5. Line integral convolution (LIC) approximation
    ax = axes[1, 1]
    # Create a simple noise texture
    np.random.seed(42)
    noise = np.random.rand(*Xi.shape)
    
    # Blur noise along streamlines (simplified LIC)
    from scipy.ndimage import gaussian_filter
    lic_approx = gaussian_filter(noise, sigma=1.0)
    
    # Apply mask for domain
    lic_approx[mask] = np.nan
    
    im = ax.imshow(lic_approx, extent=[-2, 2, -2, 2], origin='lower', 
                   cmap='gray', alpha=0.7)
    
    # Overlay some streamlines
    ax.streamplot(Xi, Yi, Bx_interp, By_interp, 
                 color='blue', density=0.8, linewidth=0.5, 
                 arrowsize=0.8)
    
    ax.set_title('LIC-style Visualization')
    ax.set_aspect('equal')
    
    # 6. Divergence and vorticity
    ax = axes[1, 2]
    
    # Compute divergence and vorticity
    dVx_dx, dVx_dy = np.gradient(Bx_interp)
    dVy_dx, dVy_dy = np.gradient(By_interp)
    
    divergence = dVx_dx + dVy_dy
    vorticity = dVy_dx - dVx_dy
    
    # Plot vorticity (should be small for magnetic field)
    vort_levels = np.linspace(-np.max(np.abs(vorticity)), np.max(np.abs(vorticity)), 20)
    vort_plot = ax.contourf(Xi, Yi, vorticity, levels=vort_levels, cmap='RdBu_r')
    fig.colorbar(vort_plot, ax=ax, label='Vorticity')
    
    # Add text annotation about divergence
    max_div = np.max(np.abs(divergence[~np.isnan(divergence)]))
    ax.text(0.02, 0.98, f'Max |∇·B|: {max_div:.2e}', 
            transform=ax.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_title('Field Vorticity (∇×B)')
    ax.set_aspect('equal')
    
    # Add domain boundary and labels to all plots
    for ax in axes.flat:
        boundary_circle = Circle((0, 0), 2.0, fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(boundary_circle)
        ax.set_xlim(-2.2, 2.2)
        ax.set_ylim(-2.2, 2.2)
        ax.set_xlabel('x [m]')
        ax.set_ylabel('y [m]')
    
    plt.tight_layout()
    return fig

# Create vector field visualizations
fig_vectors = create_vector_field_plots(points_viz, elements_viz, Bx_viz, By_viz, B_magnitude_viz)
plt.show()

print("Vector field visualizations created!")
print(f"Max divergence (should be ~0): {np.max(np.abs(np.gradient(Bx_viz)[0] + np.gradient(By_viz)[1])):.2e}")

## 2. Derived Quantity Computation

Now we'll compute physically meaningful quantities from the FEA results, including magnetic energy, flux linkage, forces, and other engineering metrics.

In [ ]:
# ---------- 2. Energy and Flux Calculations ----------
def compute_magnetic_energy(points, elements, Bx, By, mu=4*np.pi*1e-7):
    """
    Compute magnetic energy stored in the field.
    
    E = ∫(B²/2μ) dV
    
    For 2D problems, we integrate over area.
    """
    total_energy = 0.0
    element_energies = []
    
    for elem in elements:
        # Get element properties
        coords = points[elem]
        x = coords[:, 0]
        y = coords[:, 1]
        
        # Element area
        A_e = 0.5 * abs(np.linalg.det(np.array([
            [1, x[0], y[0]],
            [1, x[1], y[1]],
            [1, x[2], y[2]]
        ])))
        
        # Average B field in element
        Bx_avg = np.mean(Bx[elem])
        By_avg = np.mean(By[elem])
        B_mag_avg = np.sqrt(Bx_avg**2 + By_avg**2)
        
        # Element energy density
        energy_density = B_mag_avg**2 / (2 * mu)
        element_energy = energy_density * A_e
        
        element_energies.append(element_energy)
        total_energy += element_energy
    
    return total_energy, np.array(element_energies)

def compute_flux_linkage(points, elements, A_z, current_region_mask):
    """
    Compute flux linkage through a current-carrying region.
    
    Φ = ∫A·J dV (simplified for 2D)
    """
    flux_linkage = 0.0
    
    for i, elem in enumerate(elements):
        if current_region_mask[i]:
            # Element properties
            coords = points[elem]
            x = coords[:, 0]
            y = coords[:, 1]
            
            # Element area
            A_e = 0.5 * abs(np.linalg.det(np.array([
                [1, x[0], y[0]],
                [1, x[1], y[1]],
                [1, x[2], y[2]]
            ])))
            
            # Average A_z in element
            A_z_avg = np.mean(A_z[elem])
            
            # Contribution to flux linkage
            flux_linkage += A_z_avg * A_e
    
    return flux_linkage

def compute_force_maxwell(points, elements, Bx, By, mu=4*np.pi*1e-7, surface_points=None):
    """
    Compute electromagnetic force using Maxwell stress tensor.
    
    F = ∮T·n dS where T is Maxwell stress tensor
    """
    if surface_points is None:
        # Use a circular surface for demonstration
        surface_radius = 1.0
        surface_points = [i for i, point in enumerate(points) 
                         if abs(np.linalg.norm(point) - surface_radius) < 0.1]
    
    total_force = np.array([0.0, 0.0])
    
    # For each point on surface, compute stress
    for point_idx in surface_points:
        # Get field at surface point
        Bx_local = Bx[point_idx]
        By_local = By[point_idx]
        B_mag = np.sqrt(Bx_local**2 + By_local**2)
        
        # Normal vector (pointing outward)
        point = points[point_idx]
        normal = point / np.linalg.norm(point)
        
        # Maxwell stress tensor components (2D)
        # T_ij = (1/μ)(B_i B_j - 1/2 δ_ij B²)
        T_xx = (1/mu) * (Bx_local**2 - 0.5 * B_mag**2)
        T_xy = (1/mu) * (Bx_local * By_local)
        T_yx = T_xy
        T_yy = (1/mu) * (By_local**2 - 0.5 * B_mag**2)
        
        # Force = T · n
        force_x = T_xx * normal[0] + T_xy * normal[1]
        force_y = T_yx * normal[0] + T_yy * normal[1]
        
        # Approximate surface element size
        surface_element = 0.1  # Simplified
        
        total_force += np.array([force_x, force_y]) * surface_element
    
    return total_force

# Compute derived quantities
print("Computing derived quantities...")

# Magnetic energy
total_energy, element_energies = compute_magnetic_energy(points_viz, elements_viz, Bx_viz, By_viz)
print(f"Total magnetic energy: {total_energy:.4e} J")
print(f"Average energy per element: {total_energy/len(elements_viz):.4e} J")

# Identify high-energy regions
high_energy_threshold = np.percentile(element_energies, 90)
high_energy_elements = element_energies > high_energy_threshold
print(f"High-energy elements (top 10%): {np.sum(high_energy_elements)}")

# Flux linkage (through central region)
central_region_mask = np.array([np.mean(np.linalg.norm(points_viz[elem], axis=1)) < 0.5 
                                for elem in elements_viz])
flux_central = compute_flux_linkage(points_viz, elements_viz, A_z_viz, central_region_mask)
print(f"Flux linkage through central region: {flux_central:.4e} Wb")

# Force calculation
electromagnetic_force = compute_force_maxwell(points_viz, elements_viz, Bx_viz, By_viz)
print(f"Electromagnetic force: ({electromagnetic_force[0]:.4e}, {electromagnetic_force[1]:.4e}) N")
print(f"Force magnitude: {np.linalg.norm(electromagnetic_force):.4e} N")

# Additional field statistics
print(f"\nField Statistics:")
print(f"  Max B_x: {Bx_viz.max():.4e} T")
print(f"  Max B_y: {By_viz.max():.4e} T")
print(f"  Max |B|: {B_magnitude_viz.max():.4e} T")
print(f"  Average |B|: {B_magnitude_viz.mean():.4e} T")
print(f"  Total magnetic flux: {flux_central:.4e} Wb")

In [ ]:
# ---------- 2.1 Energy Distribution Visualization ----------
def visualize_energy_distribution(points, elements, element_energies, B_magnitude):
    """
    Create visualizations of energy distribution in the domain.
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Energy and Field Distribution Analysis', fontsize=16)

    # 1. Energy density distribution
    ax = axes[0, 0]
    energy_plot = ax.tripcolor(points[:, 0], points[:, 1], elements, 
                              element_energies, cmap='hot', shading='flat')
    fig.colorbar(energy_plot, ax=ax, label='Energy Density [J/m²]')
    ax.set_title('Magnetic Energy Density')
    ax.set_aspect('equal')
    
    # 2. Energy vs field magnitude correlation
    ax = axes[0, 1]
    
    # Compute average field magnitude per element
    element_B_mag = []
    for elem in elements:
        element_B_mag.append(np.mean(B_magnitude[elem]))
    element_B_mag = np.array(element_B_mag)
    
    ax.scatter(element_B_mag, element_energies, alpha=0.6, s=20)
    ax.set_xlabel('Average |B| [T]')
    ax.set_ylabel('Energy Density [J/m²]')
    ax.set_title('Energy vs Field Magnitude')
    ax.grid(True, alpha=0.3)
    
    # Add trend line
    z = np.polyfit(element_B_mag, element_energies, 2)
    p = np.poly1d(z)
    B_trend = np.linspace(element_B_mag.min(), element_B_mag.max(), 100)
    ax.plot(B_trend, p(B_trend), 'r-', linewidth=2, label='Quadratic fit')
    ax.legend()
    
    # 3. Cumulative energy distribution
    ax = axes[1, 0]
    
    # Sort elements by energy
    sorted_indices = np.argsort(element_energies)[::-1]  # Descending
    cumulative_energy = np.cumsum(element_energies[sorted_indices])
    total_energy = cumulative_energy[-1]
    cumulative_fraction = cumulative_energy / total_energy
    
    ax.plot(np.arange(1, len(element_energies) + 1), cumulative_fraction * 100, 'b-', linewidth=2)
    ax.axhline(y=90, color='red', linestyle='--', alpha=0.7, label='90% of total energy')
    ax.axhline(y=50, color='green', linestyle='--', alpha=0.7, label='50% of total energy')
    ax.set_xlabel('Number of Elements')
    ax.set_ylabel('Cumulative Energy [%]')
    ax.set_title('Cumulative Energy Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add annotations
    n_90_percent = np.argmax(cumulative_fraction >= 0.9) + 1
    n_50_percent = np.argmax(cumulative_fraction >= 0.5) + 1
    ax.annotate(f'90% energy\nin {n_90_percent} elements\n({100*n_90_percent/len(element_energies):.1f}% total)',
                xy=(n_90_percent, 90), xytext=(n_90_percent + 100, 70),
                arrowprops=dict(arrowstyle='->', color='red'),
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # 4. High-energy regions highlight
    ax = axes[1, 1]
    
    # Background: field magnitude
    field_bg = ax.tricontourf(points[:, 0], points[:, 1], elements, B_magnitude, 
                             levels=20, cmap='viridis', alpha=0.3)
    
    # Highlight high-energy elements
    threshold_90 = np.percentile(element_energies, 90)
    high_energy_mask = element_energies > threshold_90
    
    for i, elem in enumerate(elements):
        if high_energy_mask[i]:
            triangle = plt.Polygon(points[elem], facecolor='red', alpha=0.7, edgecolor='none')
            ax.add_patch(triangle)
    
    ax.set_title(f'High-Energy Regions (Top 10%)')
    ax.set_aspect('equal')
    
    # Add statistics
    stats_text = (f'High-energy elements: {np.sum(high_energy_mask)}\n'
                  f'Energy fraction: {100*np.sum(element_energies[high_energy_mask])/np.sum(element_energies):.1f}%\n'
                  f'Avg field magnitude: {np.mean(element_B_mag[high_energy_mask]):.4f} T')
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Add domain boundary
    for ax in axes.flat:
        if ax in [axes[0, 0], axes[1, 1]]:  # Spatial plots
            boundary_circle = Circle((0, 0), 2.0, fill=False, edgecolor='black', linewidth=1)
            ax.add_patch(boundary_circle)
            ax.set_xlim(-2.2, 2.2)
            ax.set_ylim(-2.2, 2.2)
            ax.set_xlabel('x [m]')
            ax.set_ylabel('y [m]')
    
    plt.tight_layout()
    return fig

# Create energy distribution visualizations
fig_energy = visualize_energy_distribution(points_viz, elements_viz, element_energies, B_magnitude_viz)
plt.show()

print("Energy distribution analysis complete!")
print(f"Energy concentration: {100*np.sum(element_energies[high_energy_elements])/total_energy:.1f}% in top 10% elements")

## 3. Comparative Analysis Across Solvers

Let's compare results from different solution methods, mesh resolutions, and boundary conditions to understand their impact on accuracy and performance.

In [ ]:
# ---------- 3. Solver Comparison Framework ----------
def solve_with_different_methods(mesh_resolution='coarse'):
    """
    Solve the same problem using different methods for comparison.
    """
    # Define mesh parameters based on resolution
    if mesh_resolution == 'coarse':
        n_points = 300
        n_boundary = 30
    elif mesh_resolution == 'medium':
        n_points = 600
        n_boundary = 40
    elif mesh_resolution == 'fine':
        n_points = 1000
        n_boundary = 50
    else:
        n_points = 600
        n_boundary = 40
    
    # Create test problem
    points_test, elements_test, A_z_test, Bx_test, By_test = create_test_field_data(n_points)
    B_mag_test = np.sqrt(Bx_test**2 + By_test**2)
    
    return {
        'points': points_test,
        'elements': elements_test,
        'A_z': A_z_test,
        'Bx': Bx_test,
        'By': By_test,
        'B_mag': B_mag_test,
        'resolution': mesh_resolution,
        'n_elements': len(elements_test)
    }

def compute_solution_metrics(solution_data):
    """
    Compute various metrics for solution assessment.
    """
    A_z = solution_data['A_z']
    Bx = solution_data['Bx']
    By = solution_data['By']
    B_mag = solution_data['B_mag']
    
    # Compute energy
    energy, _ = compute_magnetic_energy(solution_data['points'], 
                                        solution_data['elements'], Bx, By)
    
    # Field statistics
    metrics = {
        'energy': energy,
        'max_A': A_z.max(),
        'min_A': A_z.min(),
        'max_B': B_mag.max(),
        'avg_B': B_mag.mean(),
        'std_B': B_mag.std(),
        'max_grad_A': np.max(np.abs(np.gradient(A_z))),
        'divergence_B': np.max(np.abs(np.gradient(Bx)[0] + np.gradient(By)[1]))
    }
    
    return metrics

def interpolate_to_common_grid(solutions, grid_size=50):
    """
    Interpolate all solutions to a common grid for comparison.
    """
    # Create common grid
    xi = np.linspace(-2.0, 2.0, grid_size)
    yi = np.linspace(-2.0, 2.0, grid_size)
    Xi, Yi = np.meshgrid(xi, yi)
    
    interpolated = {}
    
    for name, solution in solutions.items():
        points = solution['points']
        A_z = solution['A_z']
        Bx = solution['Bx']
        By = solution['By']
        B_mag = solution['B_mag']
        
        # Interpolate fields
        A_z_interp = griddata(points, A_z, (Xi, Yi), method='linear', fill_value=0)
        Bx_interp = griddata(points, Bx, (Xi, Yi), method='linear', fill_value=0)
        By_interp = griddata(points, By, (Xi, Yi), method='linear', fill_value=0)
        B_mag_interp = griddata(points, B_mag, (Xi, Yi), method='linear', fill_value=0)
        
        # Mask outside domain
        mask = np.sqrt(Xi**2 + Yi**2) > 2.0
        A_z_interp[mask] = np.nan
        Bx_interp[mask] = np.nan
        By_interp[mask] = np.nan
        B_mag_interp[mask] = np.nan
        
        interpolated[name] = {
            'Xi': Xi, 'Yi': Yi,
            'A_z': A_z_interp,
            'Bx': Bx_interp,
            'By': By_interp,
            'B_mag': B_mag_interp
        }
    
    return interpolated

# Generate solutions with different resolutions
print("Generating solutions with different mesh resolutions...")
solutions = {}
for resolution in ['coarse', 'medium', 'fine']:
    print(f"  Computing {resolution} mesh solution...")
    solutions[resolution] = solve_with_different_methods(resolution)

# Compute metrics for each solution
print("\nComputing solution metrics...")
all_metrics = {}
for name, solution in solutions.items():
    all_metrics[name] = compute_solution_metrics(solution)
    print(f"  {name:8s}: Energy={all_metrics[name]['energy']:.4e} J, ")
    print(f"           Max |B|={all_metrics[name]['max_B']:.4f} T, ")
    print(f"           Elements={solution['n_elements']}")

# Interpolate to common grid
print("\nInterpolating solutions to common grid...")
interpolated_solutions = interpolate_to_common_grid(solutions)
print("Interpolation complete!")

In [ ]:
# ---------- 3.1 Comparative Visualization ----------
def create_comparison_plots(solutions, interpolated_solutions, metrics):
    """
    Create comprehensive comparison visualizations.
    """
    fig, axes = plt.subplots(3, 3, figsize=(18, 18))
    fig.suptitle('Solver Comparison Across Mesh Resolutions', fontsize=16)

    resolutions = list(solutions.keys())
    
    # Row 1: Magnetic field magnitude
    for col, resolution in enumerate(resolutions):
        ax = axes[0, col]
        interp = interpolated_solutions[resolution]
        
        im = ax.contourf(interp['Xi'], interp['Yi'], interp['B_mag'], 
                        levels=20, cmap='viridis')
        ax.set_title(f'{resolution.capitalize()} Mesh\n{n_elements:,} elements')
        ax.set_aspect('equal')
        ax.set_xlabel('x [m]')
        ax.set_ylabel('y [m]')
        
        if col == 2:
            fig.colorbar(im, ax=ax, label='|B| [T]')
    
    # Row 2: Vector field comparison
    for col, resolution in enumerate(resolutions):
        ax = axes[1, col]
        interp = interpolated_solutions[resolution]
        
        # Background magnitude
        im = ax.contourf(interp['Xi'], interp['Yi'], interp['B_mag'], 
                        levels=15, cmap='viridis', alpha=0.5)
        
        # Vector field
        skip = 3
        ax.quiver(interp['Xi'][::skip, ::skip], interp['Yi'][::skip, ::skip],
                 interp['Bx'][::skip, ::skip], interp['By'][::skip, ::skip],
                 interp['B_mag'][::skip, ::skip], cmap='plasma',
                 scale=np.nanmax(interp['B_mag'])*25, width=0.003)
        
        ax.set_title(f'{resolution.capitalize()} - Vector Field')
        ax.set_aspect('equal')
        ax.set_xlabel('x [m]')
        ax.set_ylabel('y [m]')
    
    # Row 3: Differences from fine mesh (reference)
    if 'fine' in resolutions:
        fine_interp = interpolated_solutions['fine']
        
        for col, resolution in enumerate(resolutions):
            ax = axes[2, col]
            
            if resolution == 'fine':
                # Show fine mesh as reference
                im = ax.contourf(fine_interp['Xi'], fine_interp['Yi'], 
                                fine_interp['B_mag'], levels=20, cmap='viridis')
                ax.set_title('Fine Mesh (Reference)')
            else:
                # Compute difference
                interp = interpolated_solutions[resolution]
                diff = np.abs(interp['B_mag'] - fine_interp['B_mag'])
                rel_diff = diff / (fine_interp['B_mag'] + 1e-10) * 100
                
                im = ax.contourf(interp['Xi'], interp['Yi'], rel_diff, 
                                levels=20, cmap='Reds')
                ax.set_title(f'{resolution.capitalize()} - Error\nvs Fine Mesh')
                
                if col == 2:
                    fig.colorbar(im, ax=ax, label='Relative Error [%]')
            
            ax.set_aspect('equal')
            ax.set_xlabel('x [m]')
            ax.set_ylabel('y [m]')
    
    plt.tight_layout()
    return fig

def create_metrics_comparison(metrics):
    """
    Create comparison plots for computed metrics.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Quantitative Metrics Comparison', fontsize=16)
    
    resolutions = list(metrics.keys())
    
    # 1. Energy convergence
    ax = axes[0, 0]
    energies = [metrics[res]['energy'] for res in resolutions]
    n_elements = [solutions[res]['n_elements'] for res in resolutions]
    
    ax.loglog(n_elements, energies, 'bo-', linewidth=2, markersize=8)
    ax.set_xlabel('Number of Elements')
    ax.set_ylabel('Total Magnetic Energy [J]')
    ax.set_title('Energy Convergence')
    ax.grid(True, alpha=0.3, which='both')
    
    # Add convergence rate
    if len(energies) >= 2:
        rate = np.log(energies[-1]/energies[0]) / np.log(n_elements[-1]/n_elements[0])
        ax.text(0.05, 0.95, f'Convergence rate: {rate:.2f}', 
                transform=ax.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 2. Maximum field magnitude
    ax = axes[0, 1]
    max_B = [metrics[res]['max_B'] for res in resolutions]
    
    bars = ax.bar(resolutions, max_B, color=['red', 'orange', 'green'], alpha=0.7)
    ax.set_ylabel('Maximum |B| [T]')
    ax.set_title('Maximum Field Magnitude')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, value in zip(bars, max_B):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'{value:.4f}', ha='center', va='bottom')
    
    # 3. Field statistics comparison
    ax = axes[1, 0]
    avg_B = [metrics[res]['avg_B'] for res in resolutions]
    std_B = [metrics[res]['std_B'] for res in resolutions]
    
    x_pos = np.arange(len(resolutions))
    width = 0.35
    
    bars1 = ax.bar(x_pos - width/2, avg_B, width, label='Average |B|', alpha=0.7)
    bars2 = ax.bar(x_pos + width/2, std_B, width, label='Std |B|', alpha=0.7)
    
    ax.set_xlabel('Mesh Resolution')
    ax.set_ylabel('Field Magnitude [T]')
    ax.set_title('Field Statistics')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(resolutions)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # 4. Computational efficiency metrics
    ax = axes[1, 1]
    
    # Create synthetic performance data
    compute_time = [0.5, 2.0, 8.0]  # Simulated computation times
    memory_usage = [50, 200, 800]    # Simulated memory usage in MB
    
    ax2 = ax.twinx()
    
    line1 = ax.plot(resolutions, compute_time, 'ro-', linewidth=2, markersize=8, label='Compute Time')
    line2 = ax2.plot(resolutions, memory_usage, 'bs-', linewidth=2, markersize=8, label='Memory Usage')
    
    ax.set_xlabel('Mesh Resolution')
    ax.set_ylabel('Compute Time [s]', color='red')
    ax2.set_ylabel('Memory Usage [MB]', color='blue')
    ax.set_title('Computational Performance')
    ax.tick_params(axis='y', labelcolor='red')
    ax2.tick_params(axis='y', labelcolor='blue')
    
    # Combined legend
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax.legend(lines, labels, loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

# Create comparison visualizations
print("Creating comparative analysis visualizations...")
fig_comparison = create_comparison_plots(solutions, interpolated_solutions, all_metrics)
plt.show()

fig_metrics = create_metrics_comparison(all_metrics)
plt.show()

print("Comparative analysis complete!")
print(f"\nKey findings:")
for res in ['coarse', 'medium', 'fine']:
    print(f"  {res:8s}: Energy={all_metrics[res]['energy']:.4e} J, Max B={all_metrics[res]['max_B']:.4f} T")

## 4. Advanced Visualization Techniques

Let's explore sophisticated visualization methods including 3D plots, animations, and publication-quality figures.

In [ ]:
# ---------- 4. Advanced Visualization Techniques ----------
def create_3d_visualization(points, elements, A_z, B_magnitude):
    """
    Create 3D surface plots of the magnetic fields.
    """
    fig = plt.figure(figsize=(16, 12))
    
    # 1. 3D surface of A_z
    ax1 = fig.add_subplot(221, projection='3d')
    
    # Create interpolation grid for smooth surface
    xi = np.linspace(-2, 2, 50)
    yi = np.linspace(-2, 2, 50)
    Xi, Yi = np.meshgrid(xi, yi)
    
    # Interpolate A_z to grid
    A_z_interp = griddata(points, A_z, (Xi, Yi), method='linear', fill_value=0)
    
    # Mask outside domain
    mask = np.sqrt(Xi**2 + Yi**2) > 2.0
    A_z_interp[mask] = np.nan
    
    # Create surface plot
    surf1 = ax1.plot_surface(Xi, Yi, A_z_interp, cmap='viridis', alpha=0.8)
    ax1.set_xlabel('x [m]')
    ax1.set_ylabel('y [m]')
    ax1.set_zlabel('A_z [Wb/m]')
    ax1.set_title('3D Surface: Magnetic Vector Potential')
    fig.colorbar(surf1, ax=ax1, shrink=0.5, label='A_z [Wb/m]')
    
    # 2. 3D surface of B_magnitude
    ax2 = fig.add_subplot(222, projection='3d')
    
    # Interpolate B_magnitude to grid
    B_mag_interp = griddata(points, B_magnitude, (Xi, Yi), method='linear', fill_value=0)
    B_mag_interp[mask] = np.nan
    
    surf2 = ax2.plot_surface(Xi, Yi, B_mag_interp, cmap='plasma', alpha=0.8)
    ax2.set_xlabel('x [m]')
    ax2.set_ylabel('y [m]')
    ax2.set_zlabel('|B| [T]')
    ax2.set_title('3D Surface: Magnetic Field Magnitude')
    fig.colorbar(surf2, ax=ax2, shrink=0.5, label='|B| [T]')
    
    # 3. 3D wireframe of A_z with contours
    ax3 = fig.add_subplot(223, projection='3d')
    
    # Wireframe plot
    wire = ax3.plot_wireframe(Xi[::2, ::2], Yi[::2, ::2], A_z_interp[::2, ::2], 
                            color='blue', alpha=0.3, linewidth=0.5)
    
    # Add contour lines on bottom
    ax3.contour(Xi, Yi, A_z_interp, zdir='z', offset=A_z_interp.min(), 
               cmap='viridis', alpha=0.6)
    
    ax3.set_xlabel('x [m]')
    ax3.set_ylabel('y [m]')
    ax3.set_zlabel('A_z [Wb/m]')
    ax3.set_title('3D Wireframe with Contours')
    
    # 4. Combined 3D visualization
    ax4 = fig.add_subplot(224, projection='3d')
    
    # Plot A_z as surface
    surf4 = ax4.plot_surface(Xi, Yi, A_z_interp, cmap='viridis', alpha=0.6)
    
    # Add B_magnitude as height-coded scatter
    # Sample points for scatter plot
    scatter_indices = np.random.choice(len(points), size=min(500, len(points)), replace=False)
    scatter_points = points[scatter_indices]
    scatter_B = B_magnitude[scatter_indices]
    scatter_A = A_z[scatter_indices]
    
    scatter = ax4.scatter(scatter_points[:, 0], scatter_points[:, 1], scatter_A, 
                         c=scatter_B, cmap='plasma', s=20, alpha=0.8)
    
    ax4.set_xlabel('x [m]')
    ax4.set_ylabel('y [m]')
    ax4.set_zlabel('A_z [Wb/m]')
    ax4.set_title('Combined: A_z Surface + |B| Points')
    fig.colorbar(scatter, ax=ax4, shrink=0.5, label='|B| [T]')
    
    plt.tight_layout()
    return fig

def create_publication_quality_figure(points, elements, A_z, Bx, By, B_magnitude):
    """
    Create a publication-quality figure with multiple panels.
    """
    # Set up figure with publication-quality parameters
    plt.rcParams.update({
        'font.size': 10,
        'axes.labelsize': 10,
        'axes.titlesize': 12,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 9,
        'figure.titlesize': 14
    })
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('Electromagnetic Field Analysis', fontsize=14, fontweight='bold')

    # Panel A: Magnetic vector potential
    ax = axes[0, 0]
    levels_A = np.linspace(A_z.min(), A_z.max(), 15)
    contour_A = ax.tricontourf(points[:, 0], points[:, 1], elements, A_z, 
                              levels=levels_A, cmap='RdBu_r', extend='both')
    
    # Add contour lines
    contour_lines = ax.tricontour(points[:, 0], points[:, 1], elements, A_z, 
                                 levels=7, colors='black', linewidths=0.5, alpha=0.7)
    
    # Panel label
    ax.text(0.02, 0.98, '(a)', transform=ax.transAxes, fontsize=12, 
            fontweight='bold', verticalalignment='top')
    
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')
    ax.set_title('Magnetic Vector Potential $A_z$')
    ax.set_aspect('equal')
    
    # Add colorbar
    cbar1 = fig.colorbar(contour_A, ax=ax, label='$A_z$ [Wb/m]', 
                         orientation='horizontal', pad=0.1)
    
    # Panel B: Magnetic field magnitude
    ax = axes[0, 1]
    levels_B = np.linspace(0, B_magnitude.max(), 15)
    contour_B = ax.tricontourf(points[:, 0], points[:, 1], elements, B_magnitude, 
                              levels=levels_B, cmap='viridis')
    
    ax.text(0.02, 0.98, '(b)', transform=ax.transAxes, fontsize=12, 
            fontweight='bold', verticalalignment='top')
    
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')
    ax.set_title('Magnetic Field Magnitude $|B|$')
    ax.set_aspect('equal')
    
    cbar2 = fig.colorbar(contour_B, ax=ax, label='$|B|$ [T]', 
                         orientation='horizontal', pad=0.1)
    
    # Panel C: Field lines
    ax = axes[1, 0]
    
    # Background magnitude
    contour_bg = ax.tricontourf(points[:, 0], points[:, 1], elements, B_magnitude, 
                                levels=15, cmap='viridis', alpha=0.3)
    
    # Create streamlines
    xi = np.linspace(-2, 2, 30)
    yi = np.linspace(-2, 2, 30)
    Xi, Yi = np.meshgrid(xi, yi)
    
    Bx_interp = griddata(points, Bx, (Xi, Yi), method='linear', fill_value=0)
    By_interp = griddata(points, By, (Xi, Yi), method='linear', fill_value=0)
    
    mask = np.sqrt(Xi**2 + Yi**2) > 2.0
    Bx_interp[mask] = np.nan
    By_interp[mask] = np.nan
    
    stream = ax.streamplot(Xi, Yi, Bx_interp, By_interp, 
                          color='black', density=1.2, linewidth=0.8, 
                          arrowsize=1.0, arrowsize=1.2)
    
    ax.text(0.02, 0.98, '(c)', transform=ax.transAxes, fontsize=12, 
            fontweight='bold', verticalalignment='top')
    
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')
    ax.set_title('Magnetic Field Lines')
    ax.set_aspect('equal')
    
    # Panel D: Cross-section analysis
    ax = axes[1, 1]
    
    # Extract data along x-axis
    x_axis_mask = np.abs(points[:, 1]) < 0.1
    x_axis_points = points[x_axis_mask]
    x_axis_A = A_z[x_axis_mask]
    x_axis_B = B_magnitude[x_axis_mask]
    
    sort_idx = np.argsort(x_axis_points[:, 0])
    x_sorted = x_axis_points[sort_idx, 0]
    A_sorted = x_axis_A[sort_idx]
    B_sorted = x_axis_B[sort_idx]
    
    ax2 = ax.twinx()
    
    line1 = ax.plot(x_sorted, A_sorted, 'b-', linewidth=2, label='$A_z$')
    line2 = ax2.plot(x_sorted, B_sorted, 'r-', linewidth=2, label='$|B|$')
    
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    
    ax.text(0.02, 0.98, '(d)', transform=ax.transAxes, fontsize=12, 
            fontweight='bold', verticalalignment='top')
    
    ax.set_xlabel('x [m]')
    ax.set_ylabel('$A_z$ [Wb/m]', color='blue')
    ax2.set_ylabel('$|B|$ [T]', color='red')
    ax.tick_params(axis='y', labelcolor='blue')
    ax2.tick_params(axis='y', labelcolor='red')
    ax.set_title('Field Distribution Along x-axis')
    ax.grid(True, alpha=0.3)
    
    # Combined legend
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax.legend(lines, labels, loc='upper right')
    
    # Add domain boundaries to all spatial plots
    for ax in [axes[0, 0], axes[0, 1], axes[1, 0]]:
        boundary_circle = Circle((0, 0), 2.0, fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(boundary_circle)
        ax.set_xlim(-2.2, 2.2)
        ax.set_ylim(-2.2, 2.2)
    
    plt.tight_layout()
    
    # Reset font settings
    plt.rcParams.update(plt.rcParamsDefault)
    
    return fig

# Create advanced visualizations
print("Creating advanced visualizations...")

# Use fine mesh solution for best visualization
fine_solution = solutions['fine']

# 3D visualization
fig_3d = create_3d_visualization(fine_solution['points'], fine_solution['elements'], 
                              fine_solution['A_z'], fine_solution['B_mag'])
plt.show()

# Publication-quality figure
fig_pub = create_publication_quality_figure(fine_solution['points'], fine_solution['elements'],
                                           fine_solution['A_z'], fine_solution['Bx'], 
                                           fine_solution['By'], fine_solution['B_mag'])
plt.show()

print("Advanced visualizations complete!")

## 5. Engineering Metrics and Analysis

Let's compute and analyze engineering-relevant metrics that would be useful for design and optimization of electromagnetic devices.

In [ ]:
# ---------- 5. Engineering Metrics Analysis ----------
def compute_engineering_metrics(points, elements, A_z, Bx, By, B_magnitude):
    """
    Compute comprehensive engineering metrics for electromagnetic analysis.
    """
    metrics = {}
    
    # 1. Energy metrics
    total_energy, element_energies = compute_magnetic_energy(points, elements, Bx, By)
    metrics['total_energy'] = total_energy
    metrics['energy_density_max'] = np.max(element_energies)
    metrics['energy_density_avg'] = np.mean(element_energies)
    
    # 2. Field uniformity metrics
    # Define regions of interest
    center_region = np.array([np.mean(np.linalg.norm(points[elem], axis=1)) < 0.5 
                              for elem in elements])
    
    if np.any(center_region):
        center_B = []
        for i, elem in enumerate(elements):
            if center_region[i]:
                center_B.append(np.mean(B_magnitude[elem]))
        
        center_B = np.array(center_B)
        metrics['center_field_avg'] = np.mean(center_B)
        metrics['center_field_std'] = np.std(center_B)
        metrics['center_uniformity'] = 1 - (metrics['center_field_std'] / metrics['center_field_avg'])
    
    # 3. Flux linkage metrics
    total_flux = compute_flux_linkage(points, elements, A_z, np.ones(len(elements), dtype=bool))
    metrics['total_flux'] = total_flux
    
    # 4. Field gradient metrics
    gradients = []
    for elem in elements:
        coords = points[elem]
        x = coords[:, 0]
        y = coords[:, 1]
        
        A_e = 0.5 * abs(np.linalg.det(np.array([
            [1, x[0], y[0]],
            [1, x[1], y[1]],
            [1, x[2], y[2]]
        ])))
        
        if A_e > 1e-10:
            b = np.array([y[1] - y[2], y[2] - y[0], y[0] - y[1]])
            c = np.array([x[2] - x[1], x[0] - x[2], x[1] - x[0]])
            
            grad_x = np.dot(A_z[elem], b) / (2 * A_e)
            grad_y = np.dot(A_z[elem], c) / (2 * A_e)
            grad_mag = np.sqrt(grad_x**2 + grad_y**2)
            gradients.append(grad_mag)
    
    gradients = np.array(gradients)
    metrics['gradient_max'] = np.max(gradients)
    metrics['gradient_avg'] = np.mean(gradients)
    
    # 5. Peak field locations
    peak_indices = np.where(B_magnitude > 0.9 * B_magnitude.max())[0]
    if len(peak_indices) > 0:
        peak_locations = points[peak_indices]
        metrics['peak_field_locations'] = peak_locations
        metrics['peak_field_distance_from_center'] = np.mean(np.linalg.norm(peak_locations, axis=1))
    
    # 6. Field quality metrics
    metrics['field_quality_factor'] = metrics['center_field_avg'] / metrics['gradient_avg'] if metrics['gradient_avg'] > 0 else 0
    
    # 7. Domain utilization
    active_region = B_magnitude > 0.1 * B_magnitude.max()
    metrics['domain_utilization'] = np.sum(active_region) / len(B_magnitude)
    
    return metrics

def create_engineering_dashboard(points, elements, A_z, Bx, By, B_magnitude, metrics):
    """
    Create a comprehensive engineering dashboard.
    """
    fig = plt.figure(figsize=(16, 10))
    fig.suptitle('Engineering Analysis Dashboard', fontsize=16, fontweight='bold')
    
    # Create grid for subplots
    gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.3)
    
    # Main field plot (top left, spanning 2x2)
    ax_main = fig.add_subplot(gs[0:2, 0:2])
    
    contour_main = ax_main.tricontourf(points[:, 0], points[:, 1], elements, B_magnitude, 
                                        levels=20, cmap='viridis')
    
    # Add peak field markers
    if 'peak_field_locations' in metrics:
        ax_main.scatter(metrics['peak_field_locations'][:, 0], 
                      metrics['peak_field_locations'][:, 1], 
                      c='red', s=50, marker='*', label='Peak Fields', 
                      edgecolors='white', linewidth=1)
    
    # Add center region circle
    center_circle = Circle((0, 0), 0.5, fill=False, edgecolor='white', 
                          linewidth=2, linestyle='--', alpha=0.7)
    ax_main.add_patch(center_circle)
    
    ax_main.set_title('Magnetic Field Distribution')
    ax_main.set_xlabel('x [m]')
    ax_main.set_ylabel('y [m]')
    ax_main.set_aspect('equal')
    ax_main.legend()
    
    # Key metrics (top right)
    ax_metrics = fig.add_subplot(gs[0, 2:4])
    ax_metrics.axis('off')
    
    metrics_text = (
        "KEY ENGINEERING METRICS\n" + "="*30 + "\n\n"
        f"Total Energy:        {metrics['total_energy']:.4e} J\n"
        f"Max Field Density:   {metrics['energy_density_max']:.4e} J/m²\n"
        f"Total Flux:          {metrics['total_flux']:.4e} Wb\n"
        f"Max Field:           {B_magnitude.max():.4f} T\n"
        f"Avg Field:           {B_magnitude.mean():.4f} T\n"
        f"Field Uniformity:    {metrics.get('center_uniformity', 0):.3f}\n"
        f"Quality Factor:      {metrics['field_quality_factor']:.3f}\n"
        f"Domain Utilization:  {metrics['domain_utilization']:.1%}"
    )
    
    ax_metrics.text(0.1, 0.9, metrics_text, transform=ax_metrics.transAxes, 
                   verticalalignment='top', fontfamily='monospace', fontsize=10,
                   bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    
    # Energy distribution (middle right)
    ax_energy = fig.add_subplot(gs[1, 2])
    
    # Create energy density histogram
    _, element_energies = compute_magnetic_energy(points, elements, Bx, By)
    ax_energy.hist(element_energies, bins=30, alpha=0.7, color='orange', edgecolor='black')
    ax_energy.set_xlabel('Energy Density [J/m²]')
    ax_energy.set_ylabel('Number of Elements')
    ax_energy.set_title('Energy Distribution')
    ax_energy.grid(True, alpha=0.3)
    
    # Field profile along radius (middle right, bottom)
    ax_profile = fig.add_subplot(gs[1, 3])
    
    # Radial profile
    radial_bins = np.linspace(0, 2, 20)
    radial_B = []
    radial_centers = []
    
    for i in range(len(radial_bins) - 1):
        r_inner = radial_bins[i]
        r_outer = radial_bins[i + 1]
        
        ring_mask = []
        for point in points:
            r = np.linalg.norm(point)
            if r_inner <= r < r_outer:
                ring_mask.append(True)
            else:
                ring_mask.append(False)
        
        ring_mask = np.array(ring_mask)
        if np.any(ring_mask):
            radial_B.append(np.mean(B_magnitude[ring_mask]))
            radial_centers.append((r_inner + r_outer) / 2)
    
    ax_profile.plot(radial_centers, radial_B, 'b-', linewidth=2, marker='o')
    ax_profile.set_xlabel('Radius [m]')
    ax_profile.set_ylabel('Average |B| [T]')
    ax_profile.set_title('Radial Field Profile')
    ax_profile.grid(True, alpha=0.3)
    
    # Performance indicators (bottom row)
    ax_perf = fig.add_subplot(gs[2, :])
    
    # Create performance indicators
    categories = ['Energy
Efficiency', 'Field
Uniformity', 'Domain
Utilization', 'Quality
Factor']
    values = [
        min(1.0, metrics['total_energy'] / 1e-6),  # Normalized energy
        metrics.get('center_uniformity', 0),
        metrics['domain_utilization'],
        min(1.0, metrics['field_quality_factor'] / 10)
    ]
    
    colors = ['green' if v > 0.7 else 'orange' if v > 0.4 else 'red' for v in values]
    
    bars = ax_perf.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
    ax_perf.set_ylim(0, 1)
    ax_perf.set_ylabel('Performance Score')
    ax_perf.set_title('Performance Indicators')
    ax_perf.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax_perf.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{value:.2f}', ha='center', va='bottom', fontweight='bold')
    
    # Add performance threshold line
    ax_perf.axhline(y=0.7, color='green', linestyle='--', alpha=0.5, label='Good')
    ax_perf.axhline(y=0.4, color='orange', linestyle='--', alpha=0.5, label='Acceptable')
    ax_perf.legend(loc='upper right')
    
    plt.tight_layout()
    return fig

# Compute engineering metrics
print("Computing comprehensive engineering metrics...")
engineering_metrics = compute_engineering_metrics(fine_solution['points'], fine_solution['elements'],
                                                 fine_solution['A_z'], fine_solution['Bx'], 
                                                 fine_solution['By'], fine_solution['B_mag'])

# Create engineering dashboard
fig_dashboard = create_engineering_dashboard(fine_solution['points'], fine_solution['elements'],
                                              fine_solution['A_z'], fine_solution['Bx'], 
                                              fine_solution['By'], fine_solution['B_mag'],
                                              engineering_metrics)
plt.show()

print("\nEngineering Analysis Summary:")
print(f"  Total magnetic energy: {engineering_metrics['total_energy']:.4e} J")
print(f"  Maximum energy density: {engineering_metrics['energy_density_max']:.4e} J/m²")
print(f"  Field uniformity (center): {engineering_metrics.get('center_uniformity', 0):.3f}")
print(f"  Quality factor: {engineering_metrics['field_quality_factor']:.3f}")
print(f"  Domain utilization: {engineering_metrics['domain_utilization']:.1%}")
print(f"  Peak field distance from center: {engineering_metrics.get('peak_field_distance_from_center', 0):.3f} m")

## 6. Best Practices for Post-Processing

Let's summarize the key principles and best practices for effective post-processing of electromagnetic FEA results.

In [ ]:
# ---------- 6. Best Practices Summary ----------
print("=" * 80)
print("POST-PROCESSING & VISUALIZATION: BEST PRACTICES")
print("=" * 80)

print("\n1. VISUALIZATION PRINCIPLES:")
viz_principles = [
    "Choose appropriate colormaps for field types (sequential for magnitude, diverging for potential)",
    "Include consistent scaling across comparative plots",
    "Add clear labels, units, and colorbars",
    "Use appropriate contour levels to highlight important features",
    "Combine multiple visualization techniques for comprehensive analysis"
]

for i, principle in enumerate(viz_principles, 1):
    print(f"   {i}. {principle}")

print("\n2. DERIVED QUANTITY CALCULATION:")
derived_quantities = [
    ("Magnetic Energy", "E = ∫(B²/2μ) dV", "Energy storage and efficiency"),
    ("Flux Linkage", "Φ = ∫A·J dV", "Inductance and coupling"),
    ("Force (Maxwell Stress)", "F = ∮T·n dS", "Electromagnetic forces"),
    ("Field Uniformity", "σ/μ of field in region", "Quality assessment"),
    ("Power Loss", "P = ∫J²/σ dV", "Loss calculations"),
    ("Field Gradients", "|∇B|", "Force and stress analysis")
]

print("   Quantity                 | Formula                  | Application")
print("   " + "-" * 75)
for quantity, formula, application in derived_quantities:
    print(f"   {quantity:<24} | {formula:<24} | {application}")

print("\n3. COMPARATIVE ANALYSIS GUIDELINES:")
comparison_guidelines = [
    "Use consistent interpolation grids for fair comparison",
    "Normalize results for scale-independent comparison",
    "Report both absolute and relative errors",
    "Include convergence studies with mesh refinement",
    "Consider computational cost vs. accuracy trade-offs",
    "Validate against analytical solutions when available"
]

for i, guideline in enumerate(comparison_guidelines, 1):
    print(f"   {i}. {guideline}")

print("\n4. ENGINEERING METRICS IMPORTANCE:")
metrics_importance = [
    ("Energy Efficiency", "System performance and optimization"),
    ("Field Uniformity", "Quality of magnetic field distribution"),
    ("Peak Fields", "Material limits and saturation concerns"),
    ("Force Calculations", "Mechanical design and stress analysis"),
    ("Flux Linkage", "Electromagnetic coupling and inductance"),
    ("Loss Mechanisms", "Thermal design and efficiency")
]

print("   Metric                  | Engineering Relevance")
print("   " + "-" * 55)
for metric, relevance in metrics_importance:
    print(f"   {metric:<23} | {relevance}")

print("\n5. PUBLICATION-QUALITY FIGURES:")
publication_tips = [
    "Use high-resolution output (300 DPI or higher)",
    "Ensure consistent font sizes and styles",
    "Include comprehensive captions and legends",
    "Use appropriate color schemes for colorblind accessibility",
    "Maintain aspect ratios and proper scaling",
    "Include error bars or uncertainty estimates when relevant"
]

for i, tip in enumerate(publication_tips, 1):
    print(f"   {i}. {tip}")

print("\n6. COMMON POST-PROCESSING ERRORS:")

errors_solutions = [
    ("Incorrect gradient calculation", "Use proper finite element gradient formulas"),
    ("Inconsistent units", "Maintain consistent units throughout calculations"),
    "Improper interpolation", "Use appropriate interpolation methods and handle boundaries"),
    ("Missing normalization", "Normalize results for fair comparison"),
    ("Incorrect energy calculation", "Include proper material properties and volume elements"),
    ("Visualization artifacts", "Check for masking and boundary condition effects")
]

print("   Error                              | Solution")
print("   " + "-" * 68)
for error, solution in errors_solutions:
    print(f"   {error:<35} | {solution}")

print("\n7. AUTOMATION AND SCRIPTING:")
automation_tips = [
    "• Create reusable functions for common calculations",
    "• Implement batch processing for parametric studies",
    "• Use standard naming conventions for files and variables",
    "• Include error handling and input validation",
    "• Document functions with clear parameter descriptions",
    "• Version control analysis scripts for reproducibility"
]

for tip in automation_tips:
    print(f"   {tip}")

print("\n" + "=" * 80)
print("Key Takeaway: Effective post-processing transforms numerical")
print("results into actionable engineering insights and design decisions.")
print("=" * 80)

print("\n📊 SUMMARY OF APPENDIX A CONTENT:")
print("   ✅ 01_fea_introduction.md - Python FEA ecosystem overview")
print("   ✅ 02_basic_fea_examples.ipynb - Complete NumPy/SciPy solver")
print("   ✅ 03_mesh_generation.ipynb - Comprehensive meshing techniques")
print("   ✅ 04_boundary_conditions.ipynb - Multi-material geometries")
print("   ✅ 05_post_processing.ipynb - Advanced visualization and analysis")
print("   ✅ 06_validation_techniques.ipynb - Library comparisons and validation")

print("\n🎯 READY FOR: Advanced electromagnetic FEA analysis and optimization!")

## Summary

This notebook provided comprehensive coverage of post-processing and visualization techniques for electromagnetic FEA results. We explored:

### ✅ Advanced Visualization Techniques

1. **Contour Plot Variations**: Standard, labeled, logarithmic, gradient, and multi-field overlays
2. **Vector Field Visualization**: Quiver plots, streamlines, selective vectors, and divergence analysis
3. **3D Visualization**: Surface plots, wireframes, and combined representations
4. **Publication-Quality Figures**: Professional multi-panel layouts with proper formatting

### 🔬 Derived Quantity Computation

- **Magnetic Energy**: Field energy storage and distribution analysis
- **Flux Linkage**: Electromagnetic coupling and inductance calculations
- **Force Computations**: Maxwell stress tensor and electromagnetic forces
- **Field Quality Metrics**: Uniformity, gradients, and quality factors

### 📊 Comparative Analysis Framework

- **Mesh Resolution Studies**: Convergence analysis and error assessment
- **Solver Method Comparison**: Quantitative metrics and performance analysis
- **Interpolation Techniques**: Consistent grid-based comparison methods
- **Statistical Analysis**: Field distributions and variability assessment

### 🎯 Engineering Metrics Dashboard

- **Performance Indicators**: Energy efficiency, field uniformity, domain utilization
- **Design Optimization**: Peak field locations and gradient analysis
- **Quality Assessment**: Comprehensive field quality metrics
- **Decision Support**: Engineering-relevant insights for design choices

### 💡 Critical Insights

- **Visualization quality** directly impacts understanding and communication of results
- **Derived quantities** provide essential engineering insights beyond basic field plots
- **Comparative analysis** is crucial for method validation and optimization
- **Automation and standardization** ensure reproducibility and efficiency

The post-processing techniques learned here enable comprehensive analysis of electromagnetic FEA results, supporting both research investigations and practical engineering applications. These methods transform raw numerical data into actionable insights for electromagnetic device design and optimization.

---

*Next: [Validation & Library Comparisons](06_validation_techniques.ipynb)*